# LexGuard Domain Adaptation (Lab 8)
This notebook uses the `unsloth` library to efficiently fine-tune **Mistral-7B-Instruct** on your custom HuggingFace dataset (`instruction_dataset.json`).

### Prerequisites:
1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Upload your `instruction_dataset.json` file to the Colab files pane on the left.
3. Get your HuggingFace Output Token (with Write permissions) ready.

In [ ]:
# 1. Install Dependencies (Unsloth makes training 2x faster and uses 70% less memory)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load the Model (Mistral-7B-Instruct) in 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Good for legal clauses
dtype = None # Auto-detect
load_in_4bit = True # Extremely important to prevent Colab from crashing

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit", # 4-bit optimized Mistral
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Add LoRA Adapters (This is what you are actually "training")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank of the adapter
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 4. Format the Dataset
# We wrap your dataset in Mistral's specific instruction format
mistral_prompt = """[INST] You are LexGuard, a Neuro-Symbolic Compliance Auditor.
{instruction}

Contract Clause:
{input}
[/INST]
{output}</s>"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = mistral_prompt.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
# Ensure 'instruction_dataset.json' is uploaded to Colab!
dataset = load_dataset("json", data_files="instruction_dataset.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
# 5. Train the Model (This should take ~5-10 minutes on T4 for 50 rows)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 steps is usually enough for 50 rows to overfit slightly (which is what we want here)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Push the Fine-Tuned Adapter to HuggingFace
# IMPORTANT: Replace "your-username" and "your-token" with your real HG details!
import os
HF_TOKEN = "hf_YOUR_WRITE_TOKEN_HERE"   # <--- PASTE YOUR TOKEN HERE
HF_REPO = "your-username/LexGuard-Mistral-Risk-Adapter" # <--- CHANGE THIS

model.push_to_hub(HF_REPO, token = HF_TOKEN) # Pushes only the tiny LoRA adapter (~50MB)
tokenizer.push_to_hub(HF_REPO, token = HF_TOKEN)

### Congratulations! You completed Lab 8 PEFT Domain-Adaptation!
You can now go back to VSCode, grab a HuggingFace Inference API Key, and query `your-username/LexGuard-Mistral-Risk-Adapter`!